# Forward Problem
To solve the forward problem, one needs a source space, a head model and the electrode positions. All three elements need to be coregistered.

## Imports

In [1]:
#%matplotlib qt

import mindmap as mm
import os
from pathlib import Path
import mne

mm.io.read_config("../configurations/spike.yml", env=True)

## Data

In [9]:
subject = "sub-pat22"
fs_dir  = Path(os.environ["FS_DIR"])
mm_dir  = Path(os.environ["MM_DIR"])

# files
sources    = mm_dir/subject/"src"/f"{subject}_ddm-surf-7-src.fif"
head_model = mm_dir/subject/"fwd"/f"{subject}_hm-bem.fif"
eeg        = mm_dir/subject/"eeg"/f"{subject}_ave.fif"

# load
sources    = mne.read_source_spaces(sources)
head_model = mne.read_bem_surfaces(head_model)
eeg        = mne.read_evokeds(eeg)[0]

    Reading a source space...


    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    3 BEM surfaces found
    Reading a surface...
[done]
    Reading a surface...
[done]
    Reading a surface...
[done]
    3 BEM surfaces read
Reading /home/admin/work/data/eeg_epi_spikeerp_data_1.0.0/derivatives/mindmap/sub-pat22/eeg/sub-pat22_ave.fif ...
    Found the data of interest:
        t =       0.00 ...     996.00 ms (No comment)
        0 CTF compensation matrices available
        nave = 1 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied


## Step 1: Check the coregistration

`trans` corresponds to the transformation matrix that coregisters the electrodes to the scalp. If the electrodes are already coregistered, we can replace `trans` by an identity matrix.

In [3]:
trans = mne.transforms.Transform(fro="head", to="mri")      # identity matrix

mne.viz.plot_alignment(
    eeg.info,
    trans,
    subject=subject,
    dig=True,
    subjects_dir=fs_dir,
    surfaces=["head","white","outer_skull", "inner_skull"],
)

Using pyvistaqt 3d backend.

For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
Using outer_skin.surf for head surface.
Channel types::	eeg: 204


## Step 2: Compute the conduction model

In [4]:
# (1 min)
bem = mne.make_bem_solution(head_model)

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        outer skin  (2562) -> outer skin  (2562) ...
        outer skin  (2562) -> outer skull (2562) ...
        outer skin  (2562) -> inner skull (2562) ...
        outer skull (2562) -> outer skin  (2562) ...
        outer skull (2562) -> outer skull (2562) ...
        outer skull (2562) -> inner skull (2562) ...
        inner skull (2562) -> outer skin  (2562) ...
        inner skull (2562) -> outer skull (2562) ...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix...
IP approach required...
    Matrix coefficients (homog)...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix (homog)...
    Modify the original solution to incorporate IP approach...
        Combining...
        Scaling...
Solution ready.
BEM geometry computations complete.


## Step 3: Compute the forward matrix

In [5]:
forward = mne.make_forward_solution(
    info=eeg.info,
    trans=trans,
    src=sources,
    bem=bem,
    meg=False,
    eeg=True,
    verbose=False,
)

## Save forward matrix

In [6]:
folder = mm_dir/subject/"fwd" 
if folder.exists() == False:
    folder.mkdir()

print(f"Saving forward solution for subject: {subject}")
mne.write_forward_solution(
    folder/f"{subject}_surf-fwd.fif",
    forward,
    overwrite = True,
)

Saving forward solution for subject: sub-pat02
Overwriting existing file.
    Write a source space...
    [done]
    Write a source space...
    [done]
    2 source spaces written


## Test the forward model

In [10]:
import mne
from pathlib import Path
import mindmap as mm
import pyvista as pv
import numpy as np 

# configuration
config  = mm.io.read_config("../configurations/spike.yml")
mm_dir  = Path(config["MM_DIR"])
subject = "sub-pat10"

# files 
fwd = mm_dir/subject/"fwd"/f"{subject}_surf-fwd.fif"
eeg = mm_dir/subject/"eeg"/f"{subject}_ave.fif"

# data
fwd   = mne.read_forward_solution(fwd)
eeg   = mne.read_evokeds(eeg)[0]
coord = mm.utils.get_ch_positions(eeg)

# cortical orientation constraint
fwd_fixed = mne.convert_forward_solution(
    fwd, surf_ori=True, force_fixed=True, use_cps=True,
)

# leadfield matrix
leadfield = fwd_fixed["sol"]["data"]

# extra component
split = fwd["src"][0]["nuse"]

Reading forward solution from /home/admin/work/data/eeg_epi_spikeerp_data_1.0.0/derivatives/mindmap/sub-pat10/fwd/sub-pat10_surf-fwd.fif...
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523) not available
    Read EEG forward solution (6372 sources, 204 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame
Reading /home/admin/work/data/eeg_epi_spikeerp_data_1.0.0/derivatives/mindmap/sub-pat10/eeg/sub-pat10_ave.fif ...
    Found the data of interest:
        t =       0.00 ...     996.00 ms (No comment)
        0 CTF compensation matrices available
        nave = 1 - aspect type = 100
No projector specified for this dataset. Please consider the method se

In [2]:
# electrode cap
cap = pv.PolyData(coord)
cap["highlight"] = np.zeros(len(coord))

# plot
plotter = mm.viz.plot_brain_signal(fwd["src"], np.zeros(leadfield.shape[1]), clim=[leadfield.min(), leadfield.max()])
plotter.add_mesh(cap, point_size=10, render_points_as_spheres=True, name="cap")
plotter.remove_scalar_bar(title="highlight")

# callback
def pick_electrode(picked_point):
    if picked_point is not None:
        # get index of the closest point
        closest_point_id = cap.find_closest_point(picked_point)

        # change highlighted electrode
        cap["highlight"] = np.zeros(len(coord))
        cap["highlight"][closest_point_id] = 1

        # change leadfield vector
        tmp = mm.viz.plot_brain_signal(fwd["src"], leadfield[closest_point_id])
        plotter.meshes[0]["signal"] = tmp.meshes[0]["signal"]
        plotter.meshes[1]["signal"] = tmp.meshes[1]["signal"]

        # update the plotter
        plotter.render()

plotter.enable_point_picking(callback=pick_electrode)
plotter.show()


Widget(value='<iframe src="http://localhost:41115/index.html?ui=P_0x7f794097f6e0_0&reconnect=auto" class="pyvi…

In [11]:
head_model = mm_dir/subject/"fwd"/f"{subject}_hm-bem.fif"
head_model = mne.read_bem_surfaces(head_model)

    3 BEM surfaces found
    Reading a surface...
[done]
    Reading a surface...
[done]
    Reading a surface...
[done]
    3 BEM surfaces read


In [31]:
surf, osurf = True, 1
src, osrc   = True, 1
iskull, oi  = True, 0.9
oskull, oo  = True, 0.8
skin, oskin = True, 0.7
elec, oe    = True, 1

plotter = pv.Plotter()

# surfaces
if surf == True:
    vertices, faces = fwd["src"][0]["rr"], fwd["src"][0]["tris"]
    surflh = mm.utils.surf_to_polydata(vertices, faces)

    vertices, faces = fwd["src"][1]["rr"], fwd["src"][1]["tris"]
    surfrh = mm.utils.surf_to_polydata(vertices, faces)

    plotter.add_mesh(surflh, color="white", opacity=osurf)
    plotter.add_mesh(surfrh, color="white", opacity=osurf)

# sources
if src == True:
    pointslh = fwd["src"][0]["rr"][fwd["src"][0]["vertno"]]
    pointsrh = fwd["src"][1]["rr"][fwd["src"][1]["vertno"]]

    plotter.add_mesh(pv.PolyData(pointslh), opacity=osrc, color="yellow", render_points_as_spheres=True)
    plotter.add_mesh(pv.PolyData(pointsrh), opacity=osrc, color="yellow", render_points_as_spheres=True)

# inner skull
if iskull == True:
    vertices, faces = head_model[2]["rr"], head_model[2]["tris"]
    surfiskull = mm.utils.surf_to_polydata(vertices, faces)
    plotter.add_mesh(surfiskull, opacity=oi, smooth_shading=True, color="lightgray")

if oskull == True:
    vertices, faces = head_model[1]["rr"], head_model[1]["tris"]
    surfoskull = mm.utils.surf_to_polydata(vertices, faces)
    plotter.add_mesh(surfoskull, opacity=oo, smooth_shading=True, color="burlywood")

if skin == True:
    vertices, faces = head_model[0]["rr"], head_model[0]["tris"]
    surfskin = mm.utils.surf_to_polydata(vertices, faces)
    plotter.add_mesh(surfskin, opacity=oskin, smooth_shading=True, color="lightseagreen")

if elec == True:
    pointselec = mm.utils.get_ch_positions(eeg)
    plotter.add_mesh(pv.PolyData(pointselec), render_points_as_spheres=True, color="darkorange", point_size=7)

plotter.show()

Widget(value='<iframe src="http://localhost:40007/index.html?ui=P_0x7fa111882cf0_23&reconnect=auto" class="pyv…